<a href="https://colab.research.google.com/github/Antx0x/PMP-2026/blob/main/Lab04/Tema3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Ex. 1 În documentaţia pgmpy (https://pgmpy.org/examples/Monty_Hall.html) găsiţi formularea şi implementarea cele-
brei probleme Monty Hall. Cerinţa acestui exerciţiu este să explicaţi problema şi codul din implementare.

Ești la un concurs televizat cu trei uși închise. În spatele uneia se află o mașină mult visată, iar celelalte două ascund capre. Alegi la întâmplare ușa zero și ai o șansă din trei să câștigi. Prezentatorul jocului, care știe exact unde este ascuns premiul, deschide ușa doi pentru a-ți arăta o capră. Apoi te întreabă dacă vrei să renunți la ușa ta și să alegi ușa unu. Deși pare ciudat, matematica ne spune că trebuie să accepți mereu schimbarea. Păstrând ușa ta, rămâi cu șansa inițială de $33.3\%$. Prin schimbare, șansele tale se dublează imediat la $66.7\%$.



Pentru a demonstra acest lucru folosim un cod care învață calculatorul să simuleze jocul. Mai întâi stabilim actorii principali reprezentând decizia ta, locația premiului și decizia gazdei. Apoi calculatorul primește regulile stricte ale concursului sub forma unor tabele speciale. Prin aceste reguli el află că prezentatorul este mereu obligat să evite ușa ta și ușa cu mașina. După stabilirea acestor limitări îi dăm calculatorului indiciile exacte ale rundei noastre. Îi transmitem că noi am ales inițial ușa zero și că gazda a deschis ulterior ușa doi. La final îi cerem să calculeze matematic unde este cel mai probabil să se afle premiul.



Rezultatul afișat de program confirmă pe deplin teoria contraintuitivă. Calculatorul ne arată clar că ușa ta are în continuare doar $33.3\%$ șanse să ascundă mașina. Ușa deschisă de gazdă are evident zero șanse. În schimb cealaltă ușă rămasă închisă a adunat toate probabilitățile rămase și a ajuns la exact $66.7\%$ șanse de câștig. Acest calcul demonstrează fără niciun dubiu că schimbarea ușii îți aduce mereu cel mai mare avantaj posibil în acest joc.

Ex. 2 Un medic doreşte să determine probabilitatea ca un pacient să aibă o anumită boală pulmonară (B), având în vedere
simptomele observate. Există trei variabile observabile care pot indica prezenţa bolii: tuse (T), dificultate de respiraţie
(D), radiografie anormală (X). Toate acestea sunt variabile binare.
De asemenea, ştim că:
● Dacă pacientul are boala (B), există o probabilitate ridicată să aibă tuse şi să apară anomalii la radiografie.
● Dificultatea de respiraţie (D) este dependentă atât de prezenţa bolii pulmonare (B), cât şi de prezenţa tusei (T).
Astfel, datele de probabilitate condiţionată sunt următoarele:
1. Probabilitatea apriori pentru boala pulmonară:

P(B = 1) = 0.1, P(B = 0) = 0.9;
2. Probabilitatea condiţionată pentru tuse (T) condiţionată de boală (B):
P(T = 1∣B = 1) = 0.8, P(T = 1∣B = 0) = 0.3;
3. Probabilitatea condiţionată pentru radiografie anormală (X) condiţionată de boală (B):
P(X = 1∣B = 1) = 0.9, P(X = 1∣B = 0) = 0.1;

4. Probabilitatea condiţionată pentru dificultate de respiraţie (D), condiţionată de boală (B) şi tuse (T):

P(D = 1∣B = 1,T = 1) = 0.9, P(D = 1∣B = 1,T = 0) = 0.4;
P(D = 1∣B = 0,T = 1) = 0.5, P(D = 1∣B = 0,T = 0) = 0.1.

a) Definiti modelul probabilist, folosind pgmpy, care sa descrie contextul de mai sus.
b) Medicul observă că pacientul are tuse şi dificultăţi de respiraţie. Determinaţi probabilitatea ca pacientul să aibă
boala pulmonară.

c) Ulterior se descoperă (cu alte mijloace) că pacientul nu avea boala pulmonară. Care este probabilitatea ca radio-
grafia să fi fost anormală?

d) Calculaţi cele 2 probabilităţi folosind regula lui Bayes şi incărcaţi calculul fie în Markdown/Latex, fie ca foto
la foia de lucru (indicaţie: puteţi să vă folosiţi de independenţele condiţionale rezultate din modelul de reţea
bayesiană)

In [7]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# a) Definiți modelul probabilist
model = DiscreteBayesianNetwork([('B', 'T'), ('B', 'X'), ('B', 'D'), ('T', 'D')])

# CPD pentru B (Boala)
cpd_b = TabularCPD(variable='B', variable_card=2, values=[[0.9], [0.1]])

# CPD pentru T (Tuse)
cpd_t = TabularCPD(variable='T', variable_card=2,
                   values=[[0.7, 0.2],  # P(T=0|B=0), P(T=0|B=1)
                           [0.3, 0.8]], # P(T=1|B=0), P(T=1|B=1)
                   evidence=['B'], evidence_card=[2])

# CPD pentru X (Radiografie) condiționat de B
cpd_x = TabularCPD(variable='X', variable_card=2,
                   values=[[0.9, 0.1],  # P(X=0|B=0), P(X=0|B=1)
                           [0.1, 0.9]], # P(X=1|B=0), P(X=1|B=1)
                   evidence=['B'], evidence_card=[2])

# CPD pentru D (Dificultate respirație) condiționat de B și T
cpd_d = TabularCPD(variable='D', variable_card=2,
                   values=[[0.9, 0.5, 0.6, 0.1],  # P(D=0 | B, T)
                           [0.1, 0.5, 0.4, 0.9]], # P(D=1 | B, T)
                   evidence=['B', 'T'], evidence_card=[2, 2])

model.add_cpds(cpd_b, cpd_t, cpd_x, cpd_d)
model.check_model()

infer = VariableElimination(model)

# b) Probabilitatea ca pacientul să aibă boala B=1, știind T=1 și D=1
rezultat_b = infer.query(variables=['B'], evidence={'T': 1, 'D': 1})
print("b) Probabilitatea să aibă boala, știind tuse și dificultate de respirație:")
print(rezultat_b)

# c) Probabilitatea ca radiografia să fie anormală X=1, știind că nu are boala B=0
rezultat_c = infer.query(variables=['X'], evidence={'B': 0})
print("c) Probabilitatea radiografiei anormale dacă nu are boala:")
print(rezultat_c)

b) Probabilitatea să aibă boala, știind tuse și dificultate de respirație:
+------+----------+
| B    |   phi(B) |
+======+==========+
| B(0) |   0.6522 |
+------+----------+
| B(1) |   0.3478 |
+------+----------+
c) Probabilitatea radiografiei anormale dacă nu are boala:
+------+----------+
| X    |   phi(X) |
+======+==========+
| X(0) |   0.9000 |
+------+----------+
| X(1) |   0.1000 |
+------+----------+


Calculul pentru subpunctul b): $P(B=1 \vert{} T=1, D=1)$Conform formulei lui Bayes, avem:$$P(B=1 \vert{} T=1, D=1) = \frac{P(B=1, T=1, D=1)}{P(T=1, D=1)}$$Din structura rețelei, știm că probabilitatea comună se factorizează astfel: $P(B, T, D) = P(B) \cdot P(T\vert{}B) \cdot P(D\vert{}B, T)$.   Calculăm numărătorul (cazul favorabil):$$P(B=1, T=1, D=1) = P(B=1) \cdot P(T=1\vert{}B=1) \cdot P(D=1\vert{}B=1, T=1)$$$$P(B=1, T=1, D=1) = 0.1 \cdot 0.8 \cdot 0.9 = 0.072$$Calculăm numitorul (probabilitatea totală a evidenței):Acesta este suma dintre probabilitatea calculată anterior (când $B=1$) și cea în care $B=0$.$$P(B=0, T=1, D=1) = P(B=0) \cdot P(T=1\vert{}B=0) \cdot P(D=1\vert{}B=0, T=1)$$$$P(B=0, T=1, D=1) = 0.9 \cdot 0.3 \cdot 0.5 = 0.135$$$$P(T=1, D=1) = 0.072 + 0.135 = 0.207$$Aplicăm împărțirea finală:$$P(B=1 \vert{} T=1, D=1) = \frac{0.072}{0.207} \approx 0.3478 \text{ (adică } 34.78\%)$$

c)
$$P(X=1 \vert{} B=0) = 0.1 \text{ (adică } 10\%)$$

3 Un joc între doi jucători, J0 şi J1, se desfăşoară în felul următor:
● se aruncă mai întâi cu o monedă (normală) pentru a decide cine începe: J0 sau J1;
● în prima rundă, jucătorul desemnat aruncă cu propriul zar; fie n numărul obţinut;
● în a doua rundă, celălalt jucător aruncă cu moneda proprie de 2n ori; fie m numărul de steme obţinute.
Jucătorul din prima rundă câştigă dacă n ≥ m, în caz contrar câştigând jucătorul din a doua rundă. Mai ştim că
jucătorul J1 este necinstit, el aducând o monedă măsluită, cu probabilitatea de obţinere a stemei egală cu 4/7. În schimb,
moneda jucătorului J0 este normală, iar cele două zaruri sunt şi ele normale.


a) Estimaţi care dintre cei doi jucători are şansele cele mai mari de câştig, simulând un joc de 10000 ori.

In [9]:
import numpy as np

# Numarul de simulari
N_SIMULARI = 10000

castiguri_J0 = 0
castiguri_J1 = 0

for _ in range(N_SIMULARI):

    S = np.random.choice([0, 1])

    n = np.random.randint(1, 7)

    if S == 0:
        p_stema = 4/7
    else:
        p_stema = 1/2

    m = np.random.binomial(2 * n, p_stema)

    # Jucatorul din prima runda castiga daca n >= m
    if n >= m:
        castigator = S
    else:
        castigator = 1 - S # Castiga cel din a doua runda

    # Inregistram scorul
    if castigator == 0:
        castiguri_J0 += 1
    else:
        castiguri_J1 += 1

print("=== Subpunctul a) ===")
print(f"Jucatorul J0 a castigat de {castiguri_J0} ori ({(castiguri_J0/N_SIMULARI)*100:.2f}%)")
print(f"Jucatorul J1 a castigat de {castiguri_J1} ori ({(castiguri_J1/N_SIMULARI)*100:.2f}%)")
if castiguri_J0 > castiguri_J1:
    print("Concluzie: J0 are sanse mai mari de castig.")
else:
    print("Concluzie: J1 are sanse mai mari de castig.")

=== Subpunctul a) ===
Jucatorul J0 a castigat de 4296 ori (42.96%)
Jucatorul J1 a castigat de 5704 ori (57.04%)
Concluzie: J1 are sanse mai mari de castig.


b) Folosind pgmpy, definiţi o reţea Bayesiană care sa descrie contextul de mai sus.

In [10]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from scipy.stats import binom

# Definim structura retelei: S si N influenteaza M
model = DiscreteBayesianNetwork([('S', 'M'), ('N', 'M')])

# CPD pentru S (Cine incepe)
cpd_s = TabularCPD(variable='S', variable_card=2, values=[[0.5], [0.5]])

# CPD pentru N
prob_zar = 1/6
cpd_n = TabularCPD(variable='N', variable_card=6,
                   values=[[prob_zar], [prob_zar], [prob_zar],
                           [prob_zar], [prob_zar], [prob_zar]])


valori_m = []

for m in range(13): # Numarul de steme posibile
    rand = []
    for s in [0, 1]: # Cine a inceput
        # Daca S=0, arunca J1 (p=4/7). Daca S=1, arunca J0 (p=1/2).
        p_stema = 4/7 if s == 0 else 1/2

        for n_idx in range(6):
            n_real = n_idx + 1 # Transformam starea (0-5) in valoarea zarului (1-6)
            nr_aruncari = 2 * n_real

            if m > nr_aruncari:
                # E imposibil sa ai mai multe steme decat numarul de aruncari
                prob = 0.0
            else:
                # Calculam probabilitatea folosind distributia binomiala
                prob = binom.pmf(m, nr_aruncari, p_stema)
            rand.append(prob)
    valori_m.append(rand)

cpd_m = TabularCPD(variable='M', variable_card=13, values=valori_m,
                   evidence=['S', 'N'], evidence_card=[2, 6])

model.add_cpds(cpd_s, cpd_n, cpd_m)
model.check_model()
print("\n=== Subpunctul b) ===")
print("Reteaua Bayesiana a fost definita cu succes!")


=== Subpunctul b) ===
Reteaua Bayesiana a fost definita cu succes!


c) Folosind modelul de mai sus, determinaţi cine e cel mai probabil să fi început jocul, ştiind că în a doua rundă s-a
obţinut o singură stemă.

In [11]:
from pgmpy.inference import VariableElimination

infer = VariableElimination(model)

# Calculam probabilitatea ca S=0 sau S=1 avand in vedere ca M=1 (o singura stema)
rezultat = infer.query(variables=['S'], evidence={'M': 1})

print("\n=== Subpunctul c) ===")
print("Probabilitatile pentru cine a inceput (S=0 pt J0, S=1 pt J1), stiind ca M=1:")
print(rezultat)

# Extragem valorile pentru concluzie
prob_J0 = rezultat.values[0]
prob_J1 = rezultat.values[1]

if prob_J0 > prob_J1:
    print(f"\nConcluzie: Este mai probabil sa fi inceput J0 (probabilitate {prob_J0*100:.2f}%).")
else:
    print(f"\nConcluzie: Este mai probabil sa fi inceput J1 (probabilitate {prob_J1*100:.2f}%).")


=== Subpunctul c) ===
Probabilitatile pentru cine a inceput (S=0 pt J0, S=1 pt J1), stiind ca M=1:
+------+----------+
| S    |   phi(S) |
+======+==========+
| S(0) |   0.4529 |
+------+----------+
| S(1) |   0.5471 |
+------+----------+

Concluzie: Este mai probabil sa fi inceput J1 (probabilitate 54.71%).
